## Day 3 - Part 1: RNN 기초 실습 과제
### 목표: IMDB 영화 리뷰 감성 분류 모델 만들기

이번 과제에서는 튜토리얼에서 배운 RNN, LSTM, GRU를 활용하여 

IMDB 영화 리뷰가 `긍정(positive)`인지 `부정(negative)`인지를 분류하는 텍스트 분류 모델을 직접 만들어 봅니다. 

과제 수행 절차:
1. `데이터 로드 및 전처리:` `torchtext`를 사용하여 IMDB 데이터셋을 불러오고, 텍스트를 모델이 이해할 수 있는 숫자 시퀀스로 변환합니다.

2. `모델 구축:` `nn.Embedding`, `nn.LSTM` (또는 `nn.GRU`), `nn.Linear`를 조합하여 분류 모델의 아키텍처를 완성합니다.
3. `모델 학습:` 훈련 데이터를 사용하여 모델을 학습시킵니다.
4. `모델 평가:` 테스트 데이터로 모델의 성능(정확도)을 측정합니다.

각 단계별로 `# 과제:` 또는 `# TODO:` 주석이 달린 부분을 채워주세요.

### 1. 라이브러리 임포트 및 기본 설정

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.datasets import IMDB
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### 2. 데이터 로드 및 전처리

텍스트 데이터를 처리하는 것은 숫자 데이터를 처리하는 것보다 조금 더 복잡합니다. 아래 과정을 통해 텍스트를 숫자 텐서로 변환합니다.

1. `토크나이저(Tokenizer)`: 문장을 단어 단위로 나눕니다. (예: "I love this movie" -> ["i", "love", "this", "movie"])

2. `어휘집(Vocabulary)`: 데이터셋에 있는 모든 고유한 단어에 대해 고유한 정수 인덱스를 매핑하는 사전을 만듭니다.
3. `수치화(Numericalization)`: 토큰화된 문장을 어휘집을 사용하여 정수 인덱스의 시퀀스로 변환합니다.
4. `패딩(Padding)`: 모든 문장의 길이를 동일하게 맞추기 위해, 짧은 문장의 뒤에 특정한 토큰(패딩 토큰)을 추가합니다.

In [ ]:
# 1. 토크나이저 정의
tokenizer = get_tokenizer('basic_english')

# 2. 데이터셋을 순회하며 어휘집 생성
def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

train_iter = IMDB(split='train')
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

print(f"Vocabulary size: {len(vocab)}")

# 텍스트와 레이블을 처리하는 파이프라인 함수
text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: 1 if x == 'pos' else 0

# 3. DataLoader를 위한 `collate_fn` 정의 (패딩 처리)
def collate_batch(batch):
    label_list, text_list, lengths = [], [], []
    for (_label, _text) in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        lengths.append(len(processed_text))
    
    # 과제: `torch.nn.utils.rnn.pad_sequence`를 사용하여 text_list를 패딩하세요.
    # HINT: pad_sequence는 텐서의 리스트를 입력으로 받습니다.
    # HINT: batch_first=True 옵션을 사용하여 (배치 크기, 시퀀스 길이) 형태로 만드세요.
    # HINT: padding_value는 vocab['<pad>'] 인덱스를 사용하세요.
    padded_texts = torch.nn.utils.rnn.pad_sequence(
        # TODO: 여기에 코드 작성
    )
    
    return torch.tensor(label_list, dtype=torch.float32), padded_texts, torch.tensor(lengths)

# DataLoader 생성
BATCH_SIZE = 64
train_iter, test_iter = IMDB()
train_dataloader = DataLoader(list(train_iter), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(list(test_iter), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

### 3. 모델 구축하기

이제 감성 분류를 수행할 모델을 `nn.Module`을 상속받아 직접 정의합니다.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        
        # 과제: 아래 3개의 레이어를 정의하세요.
        # 1. 임베딩 레이어 (nn.Embedding)
        #    - vocab_size: 어휘집의 크기
        #    - embed_dim: 임베딩 벡터의 차원
        self.embedding = # TODO: 여기에 코드 작성
        
        # 2. LSTM 레이어 (nn.LSTM)
        #    - input_size: embed_dim
        #    - hidden_size: hidden_dim
        #    - num_layers: n_layers
        #    - bidirectional: 양방향 여부
        #    - dropout: 드롭아웃 비율 (n_layers > 1 일 때만 적용)
        #    - batch_first=True
        self.rnn = # TODO: 여기에 코드 작성
        
        # 3. 완전연결층 (nn.Linear)
        #    - input_features: 양방향일 경우 hidden_dim * 2, 아닐 경우 hidden_dim
        #    - out_features: output_dim
        self.fc = # TODO: 여기에 코드 작성
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text shape: (batch_size, seq_len)
        
        # 1. 임베딩 적용 및 드롭아웃
        embedded = self.dropout(self.embedding(text))
        # embedded shape: (batch_size, seq_len, embed_dim)
        
        # 2. RNN/LSTM 통과
        #    - ouput: 모든 시점의 은닉 상태
        #    - hidden: 마지막 시점의 은닉 상태
        #    - cell: 마지막 시점의 셀 상태
        output, (hidden, cell) = self.rnn(embedded)
        
        # 3. 마지막 은닉 상태 추출 및 드롭아웃
        #    - 양방향(bidirectional) LSTM의 경우, 정방향 마지막 은닉 상태와 역방향 마지막 은닉 상태를 연결(concatenate)해야 합니다.
        #      hidden shape: (num_layers * num_directions, batch_size, hidden_dim)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        
        # 4. 완전연결층 통과
        return self.fc(hidden).squeeze(1)

# 하이퍼파라미터 정의
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

model = RNNClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT).to(device)

### 4. 모델 학습 및 평가 함수 정의

이진 분류 문제이므로 `BCEWithLogitsLoss`를 손실 함수로 사용합니다. 

(모델 출력에 sigmoid를 적용할 필요가 없음)

평가 지표로는 정확도(Accuracy)를 사용합니다.

In [ ]:
optimizer = optim.Adam(model.parameters())
criterion = nn.BCEWithLogitsLoss().to(device)

def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

def train(model, dataloader, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    
    for labels, texts, _ in dataloader:
        labels, texts = labels.to(device), texts.to(device)
        
        # 과제: 모델 학습의 4단계를 구현하세요.
        # 1. 기울기 초기화
        # TODO: 여기에 코드 작성
        
        # 2. 모델 예측
        predictions = # TODO: 여기에 코드 작성
        
        # 3. 손실 및 정확도 계산
        loss = # TODO: 여기에 코드 작성
        acc = binary_accuracy(predictions, labels)
        
        # 4. 역전파 및 가중치 업데이트
        # TODO: 여기에 코드 작성 (2줄)
        
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        
    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

def evaluate(model, dataloader, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()
    
    with torch.no_grad():
        for labels, texts, _ in dataloader:
            labels, texts = labels.to(device), texts.to(device)
            predictions = model(texts).squeeze(1)
            loss = criterion(predictions, labels)
            acc = binary_accuracy(predictions, labels)
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            
    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

### 5. 모델 학습 실행

이제 정의한 함수들을 사용하여 실제 학습을 진행합니다.

In [ ]:
N_EPOCHS = 5

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_dataloader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, test_dataloader, criterion)
    
    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

### 6. (선택) 심화 과제

1. `단방향(Unidirectional) vs 양방향(Bidirectional)`: `BIDIRECTIONAL` 파라미터를 `False`로 바꾸고 모델을 다시 학습시켜 보세요. 양방향 모델과 성능 차이가 얼마나 나는지 확인해 보세요. (단, `forward` 함수의 `hidden` 상태 처리 부분을 단방향에 맞게 수정해야 합니다.)

2. `RNN vs GRU`: 모델의 `nn.LSTM`을 `nn.GRU`로 바꾸고 학습을 진행해 보세요. 학습 속도와 성능에 어떤 변화가 있는지 비교해 보세요.
3. `예측 함수 만들기`: 학습된 모델을 사용하여, 임의의 영화 리뷰 문장을 입력했을 때 '긍정' 또는 '부정'으로 예측하는 함수를 만들어 보세요.